In [122]:
using LowLevelFEM, LinearAlgebra

openGeometry("I-szelveny-2.geo")

In [123]:
mat = Material("body")
rem1 = Material("P")
rem2 = Material("Q")
U = Field([mat, rem1, rem2], type=:VectorField, dim=3, fieldName=:u)
Φ = Field([mat, rem1, rem2], type=:VectorField, dim=3, fieldName=:φ)

Problem("I-szelveny-2", :VectorField, 3, 3, Material[Material("body", :Hooke, 200000.0, 0.3, 115384.61538461536, 76923.07692307692, 166666.66666666663, 7.85e-9, 45.0, 4.2e8, 1.2e-5, 1.0e-7, 0.1, 1.0), Material("P", :Hooke, 200000.0, 0.3, 115384.61538461536, 76923.07692307692, 166666.66666666663, 7.85e-9, 45.0, 4.2e8, 1.2e-5, 1.0e-7, 0.1, 1.0), Material("Q", :Hooke, 200000.0, 0.3, 115384.61538461536, 76923.07692307692, 166666.66666666663, 7.85e-9, 45.0, 4.2e8, 1.2e-5, 1.0e-7, 0.1, 1.0)], 1.0, 38188, LowLevelFEM.Geometry("", "", 0, 0, nothing, nothing, nothing, nothing), :φ, :rhs, false)

In [124]:
Ku = ∫(SymGrad(U) ⋅ D(:Solid, mat) ⋅ SymGrad(U), Ω="body")

sparse([1, 2, 3, 115, 116, 117, 520, 521, 522, 523  …  111834, 113164, 113165, 113166, 113632, 113633, 113634, 114562, 114563, 114564], [1, 1, 1, 1, 1, 1, 1, 1, 1, 1  …  114564, 114564, 114564, 114564, 114564, 114564, 114564, 114564, 114564, 114564], [348913.1280323181, 132908.75886881902, 109751.91811047174, -102013.99701622546, 36128.491051464465, 43150.595899615815, 26603.875223210613, 14542.122811670883, 34442.20855667096, 31659.09188345549  …  -151712.55269207136, -278724.6944459911, 37230.77620388905, -667424.913636958, 353602.00133675965, -9569.66722840449, -559406.8309144042, -110991.09785269514, -203803.0264524686, 4.955054472080327e6], 114564, 114564)

In [125]:
mpc1_u = MPC(master="P", slave="right", field=U)
mpc1_φ = MPC(master="P", slave="right", field=Φ)

mpc2_u = MPC(master="Q", slave="mid", field=U)
mpc2_φ = MPC(master="Q", slave="mid", field=Φ)


MultiPointConstraint("Q", "mid", Problem("I-szelveny-2", :VectorField, 3, 3, Material[Material("body", :Hooke, 200000.0, 0.3, 115384.61538461536, 76923.07692307692, 166666.66666666663, 7.85e-9, 45.0, 4.2e8, 1.2e-5, 1.0e-7, 0.1, 1.0), Material("P", :Hooke, 200000.0, 0.3, 115384.61538461536, 76923.07692307692, 166666.66666666663, 7.85e-9, 45.0, 4.2e8, 1.2e-5, 1.0e-7, 0.1, 1.0), Material("Q", :Hooke, 200000.0, 0.3, 115384.61538461536, 76923.07692307692, 166666.66666666663, 7.85e-9, 45.0, 4.2e8, 1.2e-5, 1.0e-7, 0.1, 1.0)], 1.0, 38188, LowLevelFEM.Geometry("", "", 0, 0, nothing, nothing, nothing, nothing), :φ, :rhs, false), Dict{Symbol, Bool}())

In [126]:
R1 = rigidRotationMap(mpc1_u, mpc1_φ)
R2 = rigidRotationMap(mpc2_u, mpc2_φ)

sparse([39, 39, 37, 38, 42, 42, 40, 41, 45, 45  …  50278, 50279, 50283, 50283, 50281, 50282, 50286, 50286, 50284, 50285], [37, 38, 39, 39, 40, 41, 42, 42, 43, 44  …  50280, 50280, 50281, 50282, 50283, 50283, 50284, 50285, 50286, 50286], [-100.0, 50.0, 100.0, -50.0, -100.0, -50.0, 100.0, 50.0, -90.0, 50.0  …  -97.07303603402052, -10.084066731523231, 93.15783661417464, 46.82781137529287, -93.15783661417464, -46.82781137529287, -93.15783661417466, -46.82781137529287, 93.15783661417466, 46.82781137529287], 114564, 114564)

In [127]:
Kuφ = Ku * (R1 + R2)

sparse([37, 38, 39, 790, 791, 792, 1063, 1064, 1065, 1387  …  92277, 92473, 92474, 92475, 92506, 92507, 92508, 103933, 103934, 103935], [37, 37, 37, 37, 37, 37, 37, 37, 37, 37  …  92508, 92508, 92508, 92508, 92508, 92508, 92508, 92508, 92508, 92508], [236207.03057280334, -61894.43291523057, -7.642612968325374e7, 3.4205846883004764e6, 3.431004771616457e6, 8.53926900313956e6, -124996.34630733053, 58250.58554701391, -5.704111835294131e6, 3682.9568796663807  …  -588073.6122665082, -2.06201749836308e7, -4.771543687733719e7, 7.472835818628294e6, 1.579301052392763e8, 8.309472877301612e7, -6.022888453845978e6, 35863.69207588304, -6.184230768302895e6, -1.973303491178166e7], 114564, 114564)

In [128]:
Kφ = (R1 + R2)' * Ku * (R1 + R2)

sparse([37, 38, 39, 1063, 1064, 1065, 1387, 1388, 1389, 50245  …  7818, 92275, 92276, 92277, 92473, 92474, 92475, 92506, 92507, 92508], [37, 37, 37, 37, 37, 37, 37, 37, 37, 37  …  92508, 92508, 92508, 92508, 92508, 92508, 92508, 92508, 92508, 92508], [7.642612968325374e9, -3.821306484162687e9, 2.671542470304186e7, 5.704111835294131e8, -2.5518395052631637e8, -1.5105581878888939e7, 2.620734725329278e8, -1.379334065962778e8, 743055.4850427744, 1.4440565804227296e8  …  -2.409818992879283e9, 5.55887657941072e7, 2.511422344022982e7, -5.047068488065608e9, -7.193523607508775e8, -3.46987223662035e8, -4.200522971909905e9, 5.610792585287825e8, 2.820386844511287e8, 1.860357122560383e10], 114564, 114564)

In [129]:
K = SystemMatrix([Ku Kuφ; Kuφ' Kφ])

sparse([1, 2, 3, 115, 116, 117, 520, 521, 522, 523  …  122382, 206839, 206840, 206841, 207037, 207038, 207039, 207070, 207071, 207072], [1, 1, 1, 1, 1, 1, 1, 1, 1, 1  …  207072, 207072, 207072, 207072, 207072, 207072, 207072, 207072, 207072, 207072], [348913.1280323181, 132908.75886881902, 109751.91811047174, -102013.99701622546, 36128.491051464465, 43150.595899615815, 26603.875223210613, 14542.122811670883, 34442.20855667096, 31659.09188345549  …  -2.409818992879283e9, 5.55887657941072e7, 2.511422344022982e7, -5.047068488065608e9, -7.193523607508775e8, -3.46987223662035e8, -4.200522971909905e9, 5.610792585287825e8, 2.820386844511287e8, 1.860357122560383e10], 229128, 229128)

In [130]:
bc1 = BoundaryCondition("P", field=U, ux=0, uy=0)
bc2 = BoundaryCondition("P", field=Φ, φx=0, φy=0)
bc3 = BoundaryCondition("Q", field=U, uz=0)
bc4 = BoundaryCondition("Q", field=Φ, φz=0)

fu = ∫(U ⋅ [0, 0, 0], Γ="P")
fφ = ∫(Φ ⋅ [0, 0, 1e7], Γ="P")

F = SystemVector([fu, fφ])

SystemVector([0.0; 0.0; … ; 0.0; 0.0;;], nothing, Problem[Problem("I-szelveny-2", :VectorField, 3, 3, Material[Material("body", :Hooke, 200000.0, 0.3, 115384.61538461536, 76923.07692307692, 166666.66666666663, 7.85e-9, 45.0, 4.2e8, 1.2e-5, 1.0e-7, 0.1, 1.0), Material("P", :Hooke, 200000.0, 0.3, 115384.61538461536, 76923.07692307692, 166666.66666666663, 7.85e-9, 45.0, 4.2e8, 1.2e-5, 1.0e-7, 0.1, 1.0), Material("Q", :Hooke, 200000.0, 0.3, 115384.61538461536, 76923.07692307692, 166666.66666666663, 7.85e-9, 45.0, 4.2e8, 1.2e-5, 1.0e-7, 0.1, 1.0)], 1.0, 38188, LowLevelFEM.Geometry("", "", 0, 0, nothing, nothing, nothing, nothing), :u, :rhs, false), Problem("I-szelveny-2", :VectorField, 3, 3, Material[Material("body", :Hooke, 200000.0, 0.3, 115384.61538461536, 76923.07692307692, 166666.66666666663, 7.85e-9, 45.0, 4.2e8, 1.2e-5, 1.0e-7, 0.1, 1.0), Material("P", :Hooke, 200000.0, 0.3, 115384.61538461536, 76923.07692307692, 166666.66666666663, 7.85e-9, 45.0, 4.2e8, 1.2e-5, 1.0e-7, 0.1, 1.0), Ma

In [131]:
u0, φ = solveField(K, F, support=[bc1, bc2, bc3, bc4], mpc=[mpc1_u, mpc1_φ, mpc2_u, mpc2_φ])

(VectorField(Matrix{Float64}[], [0.0006081790581701854; 0.00012669086480297014; … ; 0.478349299234856; -0.1282298527257312;;], [0.0], Int64[], 1, :v3D, Problem("I-szelveny-2", :VectorField, 3, 3, Material[Material("body", :Hooke, 200000.0, 0.3, 115384.61538461536, 76923.07692307692, 166666.66666666663, 7.85e-9, 45.0, 4.2e8, 1.2e-5, 1.0e-7, 0.1, 1.0), Material("P", :Hooke, 200000.0, 0.3, 115384.61538461536, 76923.07692307692, 166666.66666666663, 7.85e-9, 45.0, 4.2e8, 1.2e-5, 1.0e-7, 0.1, 1.0), Material("Q", :Hooke, 200000.0, 0.3, 115384.61538461536, 76923.07692307692, 166666.66666666663, 7.85e-9, 45.0, 4.2e8, 1.2e-5, 1.0e-7, 0.1, 1.0)], 1.0, 38188, LowLevelFEM.Geometry("", "", 0, 0, nothing, nothing, nothing, nothing), :u, :rhs, false)), VectorField(Matrix{Float64}[], [0.0; 0.0; … ; 0.0; 0.0;;], [0.0], Int64[], 1, :v3D, Problem("I-szelveny-2", :VectorField, 3, 3, Material[Material("body", :Hooke, 200000.0, 0.3, 115384.61538461536, 76923.07692307692, 166666.66666666663, 7.85e-9, 45.0, 4.

In [132]:
u = u0 + (R1 + R2) * φ

nodal VectorField
[0.0006081790581701854; 0.00012669086480297014; … ; 0.478349299234856; -0.1282298527257312;;]

In [133]:
showDoFResults(u, :uvec, name="displacement", factor=1);

In [134]:
u1 = nodesToElements(u, onPhysicalGroup="mid")
ε = (u1 ∘ ∇ + ∇ ∘ u1) / 2

E = mat.E
ν = mat.μ
I = unitTensor(ε)

σ = E / (1+ν) * (ε + ν / (1 - 2ν) * trace(ε) * I)

elementwise TensorField
[[-3.4923908417443066e-22; 2.700916622730397e-21; … ; 2.9158969336022117e-7; -1.85304203889182e-21;;], [3.0151182227243403e-21; -7.385396277196099e-22; … ; 2.9158969336023134e-7; -1.38939584431515e-21;;], [-1.4637078915751552e-21; 1.9139987370374557e-21; … ; 2.915896933602278e-7; 1.4637269200251152e-21;;], [-1.1011356741417545e-21; 4.40451406703949e-21; … ; 2.915896933602278e-7; -1.1011356741417545e-21;;], [0.0; 0.0; … ; 2.9158969336022557e-7; 0.0;;], [-1.6516748816851041e-21; -5.505642583799363e-22; … ; 2.9158969336022557e-7; 2.752839185354386e-21;;], [0.0; 0.0; … ; 2.9158969336022557e-7; 0.0;;], [1.6516891964488678e-21; -1.926974904329777e-21; … ; 2.915896933602267e-7; -5.505678370708772e-22;;], [-1.1011356741417545e-21; 3.3033855502796176e-21; … ; 2.9158969336022446e-7; -1.1011356741417545e-21;;], [5.505749944527591e-22; 0.0; … ; 2.915896933602289e-7; 1.6517035112126315e-21;;]  …  [2.9592448652009103e-21; 4.40451406703949e-21; … ; 2.915896933602278e-7; -5.849

In [135]:
R = nodePositionVector(U)
R = nodesToElements(R, onPhysicalGroup="mid")

n2 = normalVector(U, "mid")

σ2 = elementsToElements(σ, onPhysicalGroup="mid")

τ2 = σ2 * n2;

In [136]:
showElementResults(τ2, name="shear stress")

1

In [137]:
showDoFResults(n2, name="normal vector")

2

In [138]:
openPostProcessor()